# GPU Validation

Step-by-step validation of the brain pipeline (OWLv2 + Qwen3-VL + ApproachController) on this machine.

**How to use:** run cells top-to-bottom. Each cell is small enough to debug in isolation. If a cell fails, fix it and re-run just that cell — no need to restart the whole script. Inline images show what each model is seeing so you can spot bad reference photos / saturated detections at a glance.

Replaces `tools/test_target_finder.py` / `tools/test_vlm_scout.py` / `tools/test_approach.py` for interactive debugging. The scripts still exist for live-webcam workflows.

## Setup + GPU sanity check

In [ ]:
%matplotlib inline
import sys, time, statistics
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle

# Make brain/ importable whether you opened the notebook from repo root or tools/
REPO = Path.cwd()
if (REPO / 'brain').exists():
    pass
elif (REPO.parent / 'brain').exists():
    REPO = REPO.parent
else:
    raise RuntimeError('Run this notebook from the repo root or tools/')
sys.path.insert(0, str(REPO))

print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('\n⚠️  CUDA not available. Install the CUDA torch wheel:')
    print('    pip uninstall -y torch torchvision')
    print('    pip install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121')

## Step 1 — OWLv2 (image-conditioned detector)

Loads `google/owlv2-base-patch16-ensemble` from HuggingFace (~300 MB on first run, cached after). Cold start ~5-10 s on this machine.

In [ ]:
from brain.perception.target_finder import TargetFinder

t0 = time.monotonic()
finder = TargetFinder(model_name='google/owlv2-base-patch16-ensemble', min_sim=0.3)
print(f'loaded OWLv2 in {time.monotonic() - t0:.1f}s on {finder.device}')

### Visually inspect `references/ref.jpg`

**This is the single most common failure point.** OWLv2 builds a fingerprint from the *whole* reference image. If `ref.jpg` is 80% wall and 20% bottle, the fingerprint becomes 'wall + a bit of bottle' — and the model dutifully finds wall everywhere. Your `ref.jpg` should be ≥70% bottle, plain background.

In [ ]:
REF = REPO / 'references' / 'ref.jpg'
CTX = REPO / 'references' / 'ctx.jpg'
LIVE = REPO / 'references' / 'live.jpg'
assert REF.exists(), f'missing {REF} — run tools/capture_references.py first'

ref_bgr = cv2.imread(str(REF))
plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2RGB))
plt.title(f'references/ref.jpg  ({ref_bgr.shape[1]}×{ref_bgr.shape[0]})')
plt.axis('off')
plt.show()

print('CHECK: Does the bottle fill ≥70% of this frame against a plain background?')
print('If not, recapture: python tools/capture_references.py')

### Capture one webcam frame

Hold the bottle in front of the webcam, then run this cell. (No live loop — single snapshot, easy to re-run.)

In [ ]:
WEBCAM_INDEX = 0  # try 1 or 2 if your USB webcam isn't on 0

finder.load_reference(ref_bgr)

cap = cv2.VideoCapture(WEBCAM_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
assert cap.isOpened(), f'could not open webcam {WEBCAM_INDEX}'
for _ in range(5): cap.read()  # discard first few — autoexposure settles
ok, live_bgr = cap.read()
cap.release()
assert ok, 'frame capture failed'

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(live_bgr, cv2.COLOR_BGR2RGB))
plt.title(f'captured frame  ({live_bgr.shape[1]}×{live_bgr.shape[0]})')
plt.axis('off')
plt.show()

### Run OWLv2 on the frame

Helper that runs detection + draws boxes inline. Re-run this cell after tweaking `min_sim`.

In [ ]:
def detect_and_display(frame_bgr, min_sim=0.3, top_n=10):
    finder.min_sim = min_sim
    t0 = time.monotonic()
    detections = finder.detect(frame_bgr)
    elapsed_ms = (time.monotonic() - t0) * 1000

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    for rank, d in enumerate(detections[:top_n]):
        x1, y1, x2, y2 = d.xyxy
        color = 'lime' if rank == 0 else 'cyan'
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                               fill=False, edgecolor=color, linewidth=2))
        ax.text(x1, max(y1 - 5, 12), f'{d.confidence:.2f}', color=color, fontsize=10,
                bbox=dict(facecolor='black', alpha=0.6, pad=2))
    fps = 1000 / elapsed_ms if elapsed_ms > 0 else 0
    ax.set_title(f'min_sim={min_sim}  →  {len(detections)} dets, '
                 f'top {min(top_n, len(detections))} drawn  ·  {elapsed_ms:.0f} ms ({fps:.1f} FPS)')
    ax.axis('off')
    plt.show()
    return detections

_ = detect_and_display(live_bgr, min_sim=0.3)

### Sweep the threshold

If 0.3 saturated the image with hundreds of boxes, the threshold is too low for this reference. Crank it up:

In [ ]:
for sim in [0.4, 0.6, 0.8, 0.9]:
    print(f'\n--- min_sim = {sim} ---')
    _ = detect_and_display(live_bgr, min_sim=sim, top_n=5)

**What 'pass' looks like for Step 1:**
- ≤5 detections, top one cleanly around the bottle
- Inference time ~30-60 ms (= 15-30 FPS)
- Pick the lowest `min_sim` where false positives disappear → that's your `TARGET_MIN_SIM` for the controller

If you can never get a clean detection: recapture `ref.jpg` with a tighter crop and re-run from the 'inspect ref.jpg' cell.

## Step 2 — Qwen3-VL (vision-language scout)

Highest-risk step. Loads `Qwen/Qwen3-VL-8B-Instruct` in 4-bit (~5 GB download on first run, ~5 GB VRAM after load). Cold start 30-90 s.

**OOM fallback:** if this OOMs on a 12 GB Laptop GPU, edit `brain/perception/vlm_scout.py` to swap `load_in_4bit=True` for `load_in_8bit=True` in `BitsAndBytesConfig`, restart the kernel, and rerun.

In [ ]:
from brain.perception.vlm_scout import VLMScout

print('loading Qwen3-VL-8B-Instruct (4-bit)... ~30-90s cold start')
t0 = time.monotonic()
scout = VLMScout(load_in_4bit=True)
print(f'loaded in {time.monotonic() - t0:.1f}s')
print(f'VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB / '
      f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

### See what the VLM sees

Three images get fed to Qwen3-VL: REF (target crop), CTX (wider scene with landmarks), LIVE (current robot view). The model reasons about which landmarks in CTX appear in LIVE to decide which way to rotate.

In [ ]:
assert CTX.exists() and LIVE.exists(), 'missing ctx.jpg or live.jpg — run tools/capture_references.py'

imgs = [cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB) for p in (REF, CTX, LIVE)]
titles = ['REF (target)', 'CTX (wider scene)', 'LIVE (robot view)']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, t in zip(axes, imgs, titles):
    ax.imshow(img); ax.set_title(t); ax.axis('off')
plt.show()

### Single scout call

First call is the cold-start (~10 s). Inspect `rationale` — if it says `'parse failure'` or `'keyword fallback'`, the model emitted prose instead of clean JSON and the prompt needs tuning.

In [ ]:
live_bgr_static = cv2.imread(str(LIVE))

t0 = time.monotonic()
result = scout.scout(live_bgr_static, REF, CTX)
elapsed_ms = (time.monotonic() - t0) * 1000

print(f'latency:   {elapsed_ms:.0f} ms')
print(f'direction: {result.direction}')
print(f'rationale: {result.rationale}')

if 'fallback' in result.rationale.lower() or 'parse failure' in result.rationale.lower():
    print('\n⚠️  Parsing failed. The model emitted prose around the JSON, not clean JSON.')
    print('    Fix: edit SYSTEM_PROMPT in brain/perception/vlm_scout.py to be stricter about JSON only.')

### 5-trial latency benchmark

Trial 1 is cold-cache, trials 2-5 are steady state. Target: warm calls ≤ 1500 ms.

In [ ]:
timings = []
for i in range(5):
    t0 = time.monotonic()
    r = scout.scout(live_bgr_static, REF, CTX)
    elapsed = time.monotonic() - t0
    timings.append(elapsed)
    tag = 'cold' if i == 0 else 'warm'
    print(f'  trial {i+1}/5 [{tag}]  {elapsed*1000:.0f} ms  direction={r.direction}')

warm = timings[1:]
print(f'\ncold:        {timings[0]*1000:.0f} ms')
print(f'warm mean:   {statistics.mean(warm)*1000:.0f} ms')
print(f'warm median: {statistics.median(warm)*1000:.0f} ms')
print(f'warm max:    {max(warm)*1000:.0f} ms')

## Step 3 — Full pipeline (OWLv2 + Qwen3-VL + ApproachController)

Glue the two models with the orchestrator. `controller.step(frame)` returns one of `FORWARD / LEFT / RIGHT / STOP / SEARCH_LEFT / SEARCH_RIGHT`.

In [ ]:
from brain.control.loop import Action, ApproachController
from brain.control.action_to_pwm import pwm_for

controller = ApproachController(
    target_finder=finder,
    vlm_scout=scout,
    reference_photo=str(REF),
    reporter_photo=str(CTX),
)

ACTION_COLOR = {
    Action.FORWARD: 'lime',
    Action.LEFT: 'orange',
    Action.RIGHT: 'orange',
    Action.STOP: 'red',
    Action.SEARCH_LEFT: 'magenta',
    Action.SEARCH_RIGHT: 'magenta',
}

def run_one_step(webcam_index=WEBCAM_INDEX):
    cap = cv2.VideoCapture(webcam_index)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    for _ in range(5): cap.read()
    ok, frame = cap.read()
    cap.release()
    assert ok

    t0 = time.monotonic()
    action = controller.step(frame)
    elapsed_ms = (time.monotonic() - t0) * 1000
    detections = finder.detect(frame)
    left, right = pwm_for(action)

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    for d in detections[:3]:
        x1, y1, x2, y2 = d.xyxy
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                               fill=False, edgecolor='lime', linewidth=2))
    ax.set_title(f'{action.name}   pwm=({left:+d}, {right:+d})   '
                 f'step={elapsed_ms:.0f} ms   dets={len(detections)}',
                 fontsize=18, color=ACTION_COLOR[action])
    ax.axis('off')
    plt.show()
    return action

_ = run_one_step()

**Re-run the cell below as you move the bottle around.** Each run captures a fresh frame and shows the chosen Action.

Try these scenarios:
- bottle centered, far → expect `FORWARD` (lime)
- bottle on left → expect `LEFT` (orange)
- bottle on right → expect `RIGHT` (orange)
- bottle filling >15% of frame → expect `STOP` (red)
- bottle hidden / out of frame → expect `SEARCH_LEFT` or `SEARCH_RIGHT` (magenta), and the VLM gets called

In [ ]:
_ = run_one_step()

## Pass criteria

| Step | Pass when... |
|---|---|
| 1 — OWLv2 | ≤5 detections, top one tracks the bottle, ≥15 FPS |
| 2 — Qwen3-VL | warm latency ≤1500 ms, rationale is a real sentence (not 'fallback' / 'parse failure'), no OOM |
| 3 — Pipeline | Action label changes correctly as you move the bottle around the frame |

If all three pass: the brain is GPU-validated. Tell the Mac to update NEXT.md and move to wiring the Pi.